In [1]:
import pandas as pd
import numpy as np

county = pd.read_csv(
    "../data/processed/coverage_ratio_2024.csv",
    dtype={"fips": str}
)

county["fips"] = (
    county["fips"]
    .str.zfill(5)
)

print("Counties:", len(county))
display(
    county[
        [
            "County",
            "fips",
            "visits",
            "expected_visits",
            "coverage_ratio"
        ]
    ].head()
)

Counties: 86


,County,fips,visits,expected_visits,coverage_ratio
0,Aitkin,27001,330.916667,737.917976,0.448446
1,Anoka,27003,19465.083333,14296.800101,1.361499
2,Becker,27005,543.750000,1556.824558,0.349269
3,Beltrami,27007,1250.333333,2136.407148,0.585250
4,Benton,27009,673.083333,1790.251593,0.375971


In [2]:
adjacency_url = (
    "https://www2.census.gov/"
    "geo/docs/reference/county_adjacency/"
    "county_adjacency2024.txt"
)

adjacency = pd.read_csv(
    adjacency_url,
    sep="|",
    dtype={
        "County GEOID": str,
        "Neighbor GEOID": str
    }
)

print("Shape:", adjacency.shape)
display(adjacency.head())

Shape: (22196, 4)


,County Name,County GEOID,Neighbor Name,Neighbor GEOID
0,"Autauga County, AL",01001,"Autauga County, AL",01001
1,"Autauga County, AL",01001,"Chilton County, AL",01021
2,"Autauga County, AL",01001,"Dallas County, AL",01047
3,"Autauga County, AL",01001,"Elmore County, AL",01051
4,"Autauga County, AL",01001,"Lowndes County, AL",01085


In [3]:
mn_adjacency = adjacency[
    adjacency["County GEOID"].str.startswith("27")
].copy()

mn_adjacency = mn_adjacency[
    mn_adjacency["Neighbor GEOID"].str.startswith("27")
].copy()

mn_adjacency = mn_adjacency[
    mn_adjacency["County GEOID"]
    != mn_adjacency["Neighbor GEOID"]
].copy()

print(
    "Minnesota adjacency relationships:",
    len(mn_adjacency)
)

display(mn_adjacency.head())

Minnesota adjacency relationships: 444


,County Name,County GEOID,Neighbor Name,Neighbor GEOID
9066,"Aitkin County, MN",27001,"Carlton County, MN",27017
9067,"Aitkin County, MN",27001,"Cass County, MN",27021
9068,"Aitkin County, MN",27001,"Crow Wing County, MN",27035
9069,"Aitkin County, MN",27001,"Itasca County, MN",27061
9070,"Aitkin County, MN",27001,"Kanabec County, MN",27065


In [4]:
targets = [
    "Scott",
    "Pope",
    "Nicollet",
    "Norman",
    "Lac qui Parle"
]

target_fips = county[
    county["County"].isin(targets)
][
    [
        "County",
        "fips"
    ]
].copy()

target_neighbors = target_fips.merge(
    mn_adjacency,
    left_on="fips",
    right_on="County GEOID",
    how="left"
)

display(
    target_neighbors[
        [
            "County",
            "fips",
            "Neighbor Name",
            "Neighbor GEOID"
        ]
    ]
    .sort_values(
        ["County", "Neighbor Name"]
    )
)

,County,fips,Neighbor Name,Neighbor GEOID
0,Lac qui Parle,27073,"Big Stone County, MN",27011
1,Lac qui Parle,27073,"Chippewa County, MN",27023
2,Lac qui Parle,27073,"Swift County, MN",27151
3,Lac qui Parle,27073,"Yellow Medicine County, MN",27173
4,Nicollet,27103,"Blue Earth County, MN",27013
5,Nicollet,27103,"Brown County, MN",27015
6,Nicollet,27103,"Le Sueur County, MN",27079
7,Nicollet,27103,"Renville County, MN",27129
8,Nicollet,27103,"Sibley County, MN",27143
9,Norman,27107,"Becker County, MN",27005


In [5]:
all_hub_results = []

for _, county_row in county.iterrows():

    county_name = county_row["County"]
    county_fips = county_row["fips"]

    neighbor_fips = (
        mn_adjacency.loc[
            mn_adjacency["County GEOID"] == county_fips,
            "Neighbor GEOID"
        ]
        .dropna()
        .tolist()
    )

    pooled_fips = [
        county_fips
    ] + neighbor_fips

    pooled_data = county[
        county["fips"].isin(
            pooled_fips
        )
    ].copy()

    pooled_observed = (
        pooled_data["visits"].sum()
    )

    pooled_expected = (
        pooled_data["expected_visits"].sum()
    )

    pooled_ratio = (
        pooled_observed
        / pooled_expected
    )

    all_hub_results.append(
        {
            "County": county_name,
            "fips": county_fips,
            "original_ratio":
                county_row["coverage_ratio"],
            "number_of_neighbors":
                len(neighbor_fips),
            "pooled_ratio":
                pooled_ratio
        }
    )

all_hub = pd.DataFrame(
    all_hub_results
)

print(
    "Counties:",
    len(all_hub)
)

display(
    all_hub
    .sort_values("pooled_ratio")
    .head(15)
)

Counties: 86


,County,fips,original_ratio,number_of_neighbors,pooled_ratio
74,Stevens,27149,0.261575,6,0.290135
5,Big Stone,27011,0.381272,4,0.304107
77,Traverse,27155,0.445281,4,0.335268
25,Grant,27051,0.355061,6,0.335766
75,Swift,27151,0.335719,6,0.353834
36,Lac qui Parle,27073,0.181746,4,0.376859
34,Kittson,27069,0.859906,2,0.386008
70,Sibley,27143,0.463229,6,0.398081
20,Douglas,27041,0.288498,6,0.428483
60,Pope,27121,0.150870,6,0.437606


In [6]:
all_hub["pooled_rank"] = (
    all_hub["pooled_ratio"]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)
)

all_hub["pooled_percentile"] = (
    all_hub["pooled_ratio"]
    .rank(
        pct=True
    )
)

print(
    "Median pooled ratio:",
    all_hub["pooled_ratio"].median()
)

print(
    "Mean pooled ratio:",
    all_hub["pooled_ratio"].mean()
)

target_hub_ranks = all_hub[
    all_hub["County"].isin(targets)
][
    [
        "County",
        "original_ratio",
        "pooled_ratio",
        "pooled_rank",
        "pooled_percentile"
    ]
].sort_values(
    "pooled_rank"
)

display(target_hub_ranks)

Median pooled ratio: 0.5906031195187014
Mean pooled ratio: 0.7343895202099997


,County,original_ratio,pooled_ratio,pooled_rank,pooled_percentile
36,Lac qui Parle,0.181746,0.376859,6,0.069767
60,Pope,0.150870,0.437606,10,0.116279
53,Norman,0.168493,0.594012,44,0.511628
51,Nicollet,0.166118,0.782384,61,0.709302
68,Scott,0.111693,1.166103,71,0.825581


When the same adjacency-pooling procedure is applied to all 86 counties, Lac qui Parle and Pope remain among the lowest pooled-coverage counties, ranking 6th and 10th respectively. Norman moves to approximately the middle of the pooled distribution, while Nicollet and Scott move well above the lower tail. This suggests that neighboring-county activity may explain much of the low county-only activity in Norman, Nicollet, and Scott. Pope and Lac qui Parle remain the strongest candidates for persistent low coverage. These results are still only a diagnostic because adjacency treats every bordering county as equally accessible; the later distance-based spatial analysis will provide a more realistic test.

In [7]:
hub_output = all_hub[
    [
        "County",
        "pooled_ratio",
        "pooled_rank",
        "pooled_percentile"
    ]
].copy()

hub_output.to_csv(
    "../data/processed/hub_check_2024.csv",
    index=False
)

print(
    "Saved rows:",
    len(hub_output)
)

Saved rows: 86
